# Rol 3: Integración de Datos y Ciclo de Vida (DAMA-DMBOK)

Este notebook integra los datos generados en los roles anteriores, aplicando procesos de consolidación, análisis y mapeo al ciclo de vida de datos según el marco DAMA-DMBOK.

Se incluyen:
- Integración de múltiples fuentes de datos
- Análisis del ciclo de vida de datos
- Identificación de clientes comprometidos vs expuestos
- Visualización del impacto del ataque SUNBURST

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set(style="whitegrid")

In [ ]:
ruta = "C:/Users/lilia/OneDrive/Documentos/TRABAJO FINAL GESTION/ENTREGABLES ROL 3"

clientes = pd.read_csv(os.path.join(ruta, "clientes.csv"))
instalaciones = pd.read_csv(os.path.join(ruta, "instalaciones.csv"))
versiones = pd.read_csv(os.path.join(ruta, "versiones_software.csv"))
eventos = pd.read_csv(os.path.join(ruta, "eventos_seguridad.csv"))


In [ ]:
df = eventos.merge(instalaciones, on="instalacion_id", how="left")
df = df.merge(clientes, on="cliente_id", how="left")
df = df.merge(versiones, on="version_id", how="left")

print(df.head())
print("Dimensiones:", df.shape)

## Integración de datos

Se realizó la integración de múltiples fuentes mediante operaciones de merge en pandas, consolidando información de clientes, instalaciones, versiones de software y eventos de seguridad.

Esto permite una visión unificada del sistema afectado por el incidente SUNBURST.

In [ ]:
def mapear_ciclo_vida(tipo_evento):
    if tipo_evento in ["login", "acceso"]:
        return "Uso", "Data Security"
    elif tipo_evento == "actualizacion":
        return "Mantenimiento", "Data Maintenance"
    elif tipo_evento == "instalacion":
        return "Creacion", "Data Integration"
    elif tipo_evento == "eliminacion":
        return "Eliminacion", "Data Governance"
    elif tipo_evento == "alerta":
        return "Uso", "Data Security"
    else:
        return "Uso", "Data Management"

df[["fase_ciclo_vida", "area_dama"]] = df["tipo_evento"].apply(
    lambda x: pd.Series(mapear_ciclo_vida(x))
)

df["timestamp"] = pd.to_datetime(df["timestamp"])

## Mapeo al ciclo de vida de datos

Cada evento de seguridad fue clasificado dentro de una fase del ciclo de vida de datos, permitiendo analizar el comportamiento del sistema desde una perspectiva de gestión de datos según DAMA-DMBOK.

Esto facilita la identificación de riesgos en etapas específicas como uso, mantenimiento o creación de datos.

In [ ]:
metricas = df.groupby("fase_ciclo_vida").agg(
    total_eventos=("evento_id", "count"),
    eventos_anomalos=("es_anomalo", "sum")
).reset_index()

metricas

## Métricas por fase

Se calcularon métricas clave para evaluar el comportamiento del sistema:

- Total de eventos por fase del ciclo de vida
- Cantidad de eventos anómalos

Esto permite identificar las fases más críticas del sistema.

In [ ]:
clientes_expuestos = df[df["contiene_sunburst"] == True]["cliente_id"].nunique()

clientes_comprometidos = df[
    (df["contiene_sunburst"] == True) &
    (df["es_anomalo"] == True)
]["cliente_id"].nunique()

print("Clientes expuestos:", clientes_expuestos)
print("Clientes comprometidos:", clientes_comprometidos)

## Análisis de impacto

Se identificaron:

- Clientes expuestos: aquellos que tenían versiones comprometidas
- Clientes comprometidos: aquellos con actividad anómala

Esto refleja el comportamiento del ataque SUNBURST, donde muchos sistemas fueron afectados potencialmente, pero pocos realmente comprometidos.

In [ ]:
eventos_tiempo = df.groupby(df["timestamp"].dt.date).size()

eventos_tiempo.plot(title="Timeline del ataque")
plt.show()

In [ ]:
sns.countplot(data=df, x="fase_ciclo_vida")
plt.title("Eventos por fase del ciclo de vida")
plt.show()

In [ ]:
sns.countplot(data=df, x="severidad")
plt.title("Distribución de severidad")
plt.show()

## Conclusiones

- La mayoría de los eventos se concentran en la fase de uso del ciclo de vida.
- Se identificaron anomalías en múltiples fases, indicando posibles brechas de seguridad.
- Existe una gran diferencia entre clientes expuestos y comprometidos, alineado con el caso real del ataque SUNBURST.
- La integración de datos permitió obtener una visión completa del incidente.
- El uso del marco DAMA-DMBOK facilitó el análisis estructurado de los datos.